# NB 2.1 &mdash; Regressió lineal simple: una recta sobre un núvol de punts

**MP 5134** &mdash; UT2

*Dades: pingüins de l'arxipèlag Palmer.*

---
### Què farem avui

Al NB 1.2 vas entrenar una regressió lineal amb tres mesures dels pingüins i en
vas treure un número. Avui fem un pas enrere i ens quedem amb **una sola mesura**,
la llargada de l'aleta. Ho fem per una raó molt pràctica: amb una sola variable,
un model lineal és literalment **una recta**, i una recta es pot dibuixar i
mirar.

No hi haurà cap fórmula que calgui aprendre de memòria. Tot el que farem es pot
explicar amb un dibuix i una taula de números, i així és com ho farem.

Quan acabis hauries de saber respondre aquestes preguntes:

- Què vol dir que una recta sigui *la millor* per a unes dades?
- Com es llegeixen els dos números que aprèn un model lineal?
- Quant s'equivoca el model, dit en grams i dit en percentatge?
- Per què hi ha diverses maneres de mesurar l'error, i quan convé cadascuna?

## 1. Les dades

Carreguem el mateix fitxer de pingüins del NB 1.1 i ens quedem només amb les dues
columnes que farem servir: la llargada de l'aleta, que serà l'entrada, i la massa
corporal, que és el que volem predir.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Còpia de les dades de palmerpenguins (Gorman et al., 2014) al repositori del mòdul
URL_DADES = "https://raw.githubusercontent.com/pprohenspolitecnicllevant/disseny-avaluacio-models-ml/refs/heads/main/UT01-Entorn_de_treball_primer_model/penguins/penguins.csv"

df = pd.read_csv(URL_DADES)
data = df[["flipper_length_mm", "body_mass_g"]].dropna()

print(f"Pingüins amb les dues mesures: {len(data)}")
data.head()

Tenim 342 pingüins: els 344 del fitxer menys els dos que no tenen cap mesura, que
ja vas trobar al NB 1.1.

Per què l'aleta i no el bec? Al NB 1.2 ja ho vam intuir llegint els coeficients:
l'aleta és una mesura de la **mida general** de l'animal, i per això és la que
més diu sobre quant pesa.

## 2. Mirar abans de modelar

Abans d'entrenar res, dibuixem les dues columnes l'una contra l'altra. Cada punt
és un pingüí.

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(data["flipper_length_mm"], data["body_mass_g"], alpha=0.5)
plt.xlabel("Llargada de l'aleta (mm)")
plt.ylabel("Massa (g)")
plt.title("Com més llarga l'aleta, més pesa el pingüí")
plt.show()

El dibuix ja ens diu gairebé tot el que necessitem saber:

**Els punts pugen junts.** Els pingüins amb l'aleta llarga tendeixen a pesar més.
Si haguessis de resumir aquest núvol amb un sol traç de retolador, faries una
línia recta que puja cap a la dreta.

**Però el núvol té gruix.** Per a una mateixa llargada d'aleta, per exemple
200 mm, n'hi ha que pesen poc més de 3.300 g i d'altres que en pesen 4.500. Cap recta no passarà per
tots els punts. Aquest gruix és un error que **cap model no podrà eliminar
mentre només li donem l'aleta**: la massa també depèn de coses que el model no
veu, com el sexe o l'espècie. Fer servir aquestes altres columnes és feina de la
UT3.

Queda't amb aquesta idea, perquè és la més important del notebook: **el model no
buscarà una recta perfecta, perquè no existeix; buscarà la que s'equivoca menys.**

## 3. Separar entrenament i prova

Com al NB 1.2, apartem un 20% dels pingüins abans de fer res més. Aquests pingüins
no els mirarem fins a la secció 6, quan vulguem saber com de bé funciona el model
amb animals que no ha vist mai.

In [ ]:
from sklearn.model_selection import train_test_split

X = data[["flipper_length_mm"]]   # doble claudàtor: una TAULA d'una columna
y = data["body_mass_g"]           # claudàtor simple: una COLUMNA

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Pingüins per entrenar: {len(X_train)}")
print(f"Pingüins per provar:   {len(X_test)}")

Fixa't en el **doble claudàtor** de `X`. Encara que només tinguem una
característica, scikit-learn vol que `X` sigui sempre una taula (files i
columnes), perquè està pensat per treballar amb moltes característiques alhora.
Amb `data[["flipper_length_mm"]]` obtens una taula d'una sola columna; amb
`data["flipper_length_mm"]` obtindries una columna solta i `fit()` es queixaria.

`y`, en canvi, sí que és una columna solta. És el mateix conveni de la UT1: `X`
en majúscula és una taula, `y` en minúscula és un vector.

## 4. Traçar rectes a mà

Abans de deixar que scikit-learn busqui la recta, provem-ho nosaltres. Així
entendràs què és el que fa `fit()` quan li ho demanem.

Una recta queda definida per **dos números**:

- **L'ordenada a l'origen**: on comença la recta, el valor de partida.
- **El pendent**: quant puja la recta per cada mil·límetre d'aleta que avancem.

Com que ja saps programar, ho escrivim com una funció:

In [ ]:
def predir_amb_recta(aleta, pendent, ordenada):
    """Massa predita per a una llargada d'aleta, fent servir una recta."""
    return ordenada + pendent * aleta

# Dues rectes triades a ull, per provar
PENDENT_A, ORDENADA_A = 40, -3900
PENDENT_B, ORDENADA_B = 60, -7800

aletes = np.linspace(170, 235, 100)

plt.figure(figsize=(7, 5))
plt.scatter(X_train["flipper_length_mm"], y_train, alpha=0.4, label="pingüins d'entrenament")
plt.plot(aletes, predir_amb_recta(aletes, PENDENT_A, ORDENADA_A), color="tab:orange", label="recta A")
plt.plot(aletes, predir_amb_recta(aletes, PENDENT_B, ORDENADA_B), color="tab:green", label="recta B")
plt.xlabel("Llargada de l'aleta (mm)")
plt.ylabel("Massa (g)")
plt.legend()
plt.show()

Les dues rectes travessen el núvol, però són ben diferents: la B puja molt més
de pressa que la A. **Quina és millor?**

A ull no es pot dir. Necessitem un número que mesuri com s'equivoca cada recta.

### 4.1 L'error d'una predicció

Comencem pel més senzill: mirem cinc pingüins i comparem la massa real amb la que
diu la recta A.

In [ ]:
mostra = X_train.head(5).copy()
mostra["massa_real"] = y_train.head(5)
mostra["predit_A"] = predir_amb_recta(mostra["flipper_length_mm"], PENDENT_A, ORDENADA_A)
mostra["error_A"] = mostra["predit_A"] - mostra["massa_real"]
mostra

L'**error** d'una predicció és simplement la diferència entre el que diu el model
i el que és de veritat. El signe ens diu cap a on ens equivoquem: un error
positiu vol dir que la recta s'ha passat, i un de negatiu, que s'ha quedat curta.

Per decidir quina recta és millor, però, el signe ens molesta. Si un pingüí té un
error de +300 g i un altre de &minus;300 g, fer-ne la mitjana donaria zero, com si
la recta fos perfecta. Per això ens quedem amb **la mida de l'error sense el
signe**, el que en matemàtiques s'anomena *valor absolut* i en pandas és
`.abs()`.

In [ ]:
def error_mitja(y_real, y_predit):
    """De mitjana, quants grams s'equivoca una predicció."""
    return (y_predit - y_real).abs().mean()

pred_A = predir_amb_recta(X_train["flipper_length_mm"], PENDENT_A, ORDENADA_A)
pred_B = predir_amb_recta(X_train["flipper_length_mm"], PENDENT_B, ORDENADA_B)

print(f"Recta A: s'equivoca de mitjana {error_mitja(y_train, pred_A):.0f} g")
print(f"Recta B: s'equivoca de mitjana {error_mitja(y_train, pred_B):.0f} g")

Ara sí que podem comparar. Aquest número té nom propi: és l'**error absolut
mitjà**, que trobaràs escrit com **MAE** (de l'anglès *Mean Absolute Error*). Es
llegeix així: *de mitjana, aquesta recta s'equivoca tants grams per pingüí*.

La recta A guanya, però per poc: dues rectes molt diferents s'equivoquen gairebé
igual. Això ens diu que cap de les dues és gaire bona i que n'hi ha d'haver una
de millor entremig.

### 4.2 Buscar la millor recta a força de provar

Podríem anar provant pendents a mà fins a trobar la millor recta. Fem-ho amb un
bucle, que és el mateix però més ràpid.

Hi ha un truc per no haver de provar també totes les ordenades: **la millor recta
sempre passa pel centre del núvol**, el punt que té l'aleta mitjana i la massa
mitjana. Així, per a cada pendent que provem, ja sabem quina ordenada li toca.

In [ ]:
aleta_mitjana = X_train["flipper_length_mm"].mean()
massa_mitjana = y_train.mean()

resultats = []
for pendent in range(20, 81, 5):
    ordenada = massa_mitjana - pendent * aleta_mitjana   # perquè passi pel centre del núvol
    pred = predir_amb_recta(X_train["flipper_length_mm"], pendent, ordenada)
    resultats.append({"pendent": pendent, "error_mitja_g": error_mitja(y_train, pred)})

resultats = pd.DataFrame(resultats)

plt.figure(figsize=(7, 4))
plt.plot(resultats["pendent"], resultats["error_mitja_g"], marker="o")
plt.xlabel("Pendent provat (grams per mm d'aleta)")
plt.ylabel("Error mitjà (g)")
plt.title("Hi ha un pendent que s'equivoca menys que tots els altres")
plt.show()

millor = resultats.loc[resultats["error_mitja_g"].idxmin()]
print(f"Millor pendent provat: {millor['pendent']:.0f} g/mm, amb un error mitjà de {millor['error_mitja_g']:.0f} g")

La gràfica té forma de vall. Si el pendent és massa suau, la recta queda
aplanada i s'equivoca molt; si és massa fort, també. Al fons de la vall hi ha el
pendent que s'equivoca menys: al voltant de **50 grams per mil·límetre**.

**Això és exactament el que vol dir *entrenar* un model lineal**: buscar els dos
números que fan l'error tan petit com sigui possible. Nosaltres hem provat
tretze pendents; scikit-learn ho fa de manera molt més intel·ligent i troba el
valor exacte en mil·lèsimes de segon.

## 5. Que la recta la busqui scikit-learn

In [ ]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train, y_train)

pendent = model.coef_[0]
ordenada = model.intercept_

print(f"Pendent après:             {pendent:.2f} g per mm d'aleta")
print(f"Ordenada a l'origen apresa: {ordenada:.0f} g")
print(f"Error mitjà sobre l'entrenament: {error_mitja(y_train, model.predict(X_train)):.0f} g")

Scikit-learn ha trobat un pendent de **49,85**, pràcticament el mateix que el
nostre bucle, i un error mitjà idèntic. No hi ha màgia: `fit()` fa el mateix que
hem fet a mà, però sense haver de provar valors a les palpentes.

Recorda la taula del NB 1.2: el pendent i l'ordenada són **paràmetres**, els
números que el model aprèn tot sol durant `fit()`. Per això viuen en atributs que
acaben amb guió baix: `coef_` i `intercept_`.

In [ ]:
aletes = np.linspace(170, 235, 100)
aletes_df = pd.DataFrame({"flipper_length_mm": aletes})

plt.figure(figsize=(7, 5))
plt.scatter(X_train["flipper_length_mm"], y_train, alpha=0.4, label="pingüins d'entrenament")
plt.plot(aletes, model.predict(aletes_df), color="tab:red", linewidth=2, label="recta apresa")
plt.xlabel("Llargada de l'aleta (mm)")
plt.ylabel("Massa (g)")
plt.legend()
plt.show()

### 5.1 Llegir el model en veu alta

Tot el model són aquests dos números, i es poden dir amb paraules:

> **Per cada mil·límetre més d'aleta, el model hi suma uns 50 grams.**

És una frase que entendria qualsevol biòleg, i aquest és un dels grans avantatges
de la regressió lineal: és un model que **s'explica sol**. Quan a la UT5 vegis
models molt més potents, com els Random Forests o les Neural Networks en un altre mòdul, trobaràs a faltar aquesta claredat.

L'ordenada a l'origen és una altra història: diu que un pingüí amb l'aleta de
0 mm pesaria gairebé **menys sis quilos**. Mira què passa si demanem prediccions
fora de les mides que el model ha vist:

In [ ]:
aletes_a_provar = pd.DataFrame({"flipper_length_mm": [0, 150, 190, 210, 230, 300]})
aletes_a_provar["massa_predita_g"] = model.predict(aletes_a_provar).round(0)
aletes_a_provar

Entre 190 i 230 mm, les prediccions són creïbles. A 0 mm i a 150 mm surten
disbarats, i a 300 mm surt un pingüí de nou quilos que no existeix.

El motiu és que als nostres pingüins **l'aleta va de 172 a 231 mm**. Dins d'aquest
rang, la recta s'ha ajustat a dades reals. Fora, el model continua dibuixant la
recta perquè no sap fer res més, però ja no hi ha cap dada que la sostingui.
Predir fora del rang de les dades d'entrenament s'anomena **extrapolar**, i és una
de les maneres més habituals de fer dir bestieses a un model que funcionava bé.

La regla pràctica: **un model només és de fiar dins del tipus de dades amb què
l'has entrenat**.

## 6. Com s'equivoca el model amb pingüins nous

Fins ara hem mirat l'error sobre els pingüins d'entrenament. Però el que ens
interessa de veritat és com funcionarà el model amb pingüins que no ha vist mai.
Ara és el moment d'obrir el conjunt de prova.

### 6.1 Primer, casos concrets

Com vam dir a la UT1, abans de mirar cap número global mirem uns quants casos.

In [ ]:
pred_test = model.predict(X_test)

comparacio = pd.DataFrame({
    "aleta_mm": X_test["flipper_length_mm"].values,
    "real_g": y_test.values,
    "predit_g": pred_test.round(0),
})
comparacio["error_g"] = comparacio["predit_g"] - comparacio["real_g"]
comparacio["error_%"] = (100 * comparacio["error_g"].abs() / comparacio["real_g"]).round(1)
comparacio.head(10)

Hi ha pingüins on el model l'encerta per menys de trenta grams i d'altres on
s'equivoca en més de quatre-cents. L'última columna expressa el mateix error en
percentatge de la massa real.

Ara resumim aquests 69 errors en un sol número. Veurem quatre maneres de fer-ho,
perquè **cadascuna respon una pregunta diferent**.

### 6.2 L'error absolut mitjà (MAE)

És el que ja hem fet servir per comparar rectes. Respon la pregunta més
natural: *de mitjana, quants grams s'equivoca?*

El calculem a mà i amb la funció de scikit-learn, per veure que és el mateix.

In [ ]:
from sklearn.metrics import mean_absolute_error

print(f"MAE calculat a mà:        {error_mitja(y_test, pred_test):.1f} g")
print(f"MAE segons scikit-learn:  {mean_absolute_error(y_test, pred_test):.1f} g")

Uns **288 grams** d'error de mitjana. El MAE té una gran virtut: està en les
mateixes unitats que el que prediem, i per això es pot explicar a qualsevol
persona sense cap preparació: *el model s'equivoca uns tres-cents grams per
pingüí*.

### 6.3 La desviació percentual

Tres-cents grams, és molt o poc? Depèn de què pesis. En un pingüí de quatre
quilos és poca cosa; en un pollet de mig quilo seria un desastre. La desviació
percentual respon la pregunta: *de mitjana, en quin percentatge s'equivoca?*

In [ ]:
from sklearn.metrics import mean_absolute_percentage_error

desviacio_a_ma = (100 * (pred_test - y_test).abs() / y_test).mean()

print(f"Desviació percentual a mà:       {desviacio_a_ma:.1f} %")
print(f"Desviació segons scikit-learn:   {100 * mean_absolute_percentage_error(y_test, pred_test):.1f} %")

Un **7%**. És la mètrica que se sol donar quan el resultat s'ha de comunicar a
algú que no és tècnic, perquè no depèn de les unitats.

Compte amb dues coses. La primera: scikit-learn la retorna com a fracció (0,071)
i som nosaltres qui la multipliquem per 100. La segona és més important: **si els
valors reals poden ser zero o molt petits, aquesta mètrica es torna boja**, perquè
dividim per un número minúscul. Ho comprovaràs en un dels exercicis del NB 2.2,
amb temperatures.

### 6.4 Quan els errors grans fan més mal: RMSE

Imagina dos models que prediuen la massa de quatre pingüins:

- El **model 1** s'equivoca 100 g amb cadascun.
- El **model 2** n'encerta tres exactament i s'equivoca 400 g amb el quart.

Tots dos tenen el mateix MAE, però no són igual de bons per a tothom. Si
estiguessis calculant la dosi d'un medicament a partir del pes, preferiries mil
vegades el model 1: molts errors petits abans que un d'enorme.

In [ ]:
errors_model_1 = pd.Series([100, 100, 100, 100])
errors_model_2 = pd.Series([0, 0, 0, 400])

for nom, errors in [("model 1", errors_model_1), ("model 2", errors_model_2)]:
    mae = errors.abs().mean()
    rmse = np.sqrt((errors ** 2).mean())
    print(f"{nom:18}  MAE = {mae:.0f} g   RMSE = {rmse:.0f} g")

El **RMSE** (*Root Mean Squared Error*) sí que els distingeix. El truc és que,
abans de fer la mitjana, **multiplica cada error per ell mateix**. Un error de 100
es converteix en 10.000, però un de 400 es converteix en 160.000: els errors grans
passen a pesar moltíssim més. Al final es fa l'arrel quadrada per tornar a tenir
grams.

No cal que et quedis amb el càlcul. Queda't amb com es llegeix:

- **El RMSE sempre surt igual o més gran que el MAE.**
- **Com més separats estiguin, més errors grans hi ha amagats** entre els petits.

Per cert: quan scikit-learn busca *la millor recta* a `fit()`, el que fa és
minimitzar precisament aquests errors al quadrat. Per això la seva recta i la del
nostre bucle, que minimitzava el MAE, no són idèntiques al mil·límetre.

In [ ]:
from sklearn.metrics import mean_squared_error

rmse_test = np.sqrt(mean_squared_error(y_test, pred_test))
print(f"MAE:  {mean_absolute_error(y_test, pred_test):.0f} g")
print(f"RMSE: {rmse_test:.0f} g")

Els nostres pingüins donen un RMSE de 356 g contra un MAE de 288 g. La diferència
existeix però no és exagerada: hi ha alguns pingüins amb errors grans, però no
cap cas desastrós que ho domini tot.

#### D'on surt la diferència entre el RMSE i el MAE?

Anem a buscar aquests pingüins amb errors grans. Ordenem els 69 errors del
conjunt de prova de més gran a més petit, sense signe, i els dibuixem com a
barres. Hi afegim el MAE i el RMSE com a dues línies horitzontals.

In [ ]:
errors_test = (pred_test - y_test).abs().sort_values(ascending=False)

plt.figure(figsize=(9, 4))
plt.bar(range(len(errors_test)), errors_test, color="tab:gray")
plt.axhline(errors_test.mean(), color="tab:blue", linestyle="--", label=f"MAE = {errors_test.mean():.0f} g")
plt.axhline(rmse_test, color="tab:red", linestyle="--", label=f"RMSE = {rmse_test:.0f} g")
plt.xlabel("Pingüins de prova, del que s'equivoca més al que s'equivoca menys")
plt.ylabel("Error sense signe (g)")
plt.title("Uns quants pingüins s'equivoquen molt més que la resta")
plt.legend()
plt.show()

La majoria de barres queden per sota dels 600 g, però a l'esquerra n'hi ha
**quatre que passen dels 750 g**. Són aquests quatre pingüins els que estiren el
RMSE cap amunt i el separen del MAE.

Quant pesen, aquests quatre, dins de cada mètrica? El MAE suma els errors tal com
són; el RMSE suma els errors multiplicats per ells mateixos. Comparem quina part
de cada suma és culpa seva.

In [ ]:
pitjors = errors_test.head(4)

print(f"Els 4 pitjors són el {100 * len(pitjors) / len(errors_test):.0f}% dels pingüins de prova.")
print(f"Dins el MAE, pesen el  {100 * pitjors.sum() / errors_test.sum():.0f}% de la suma d'errors.")
print(f"Dins el RMSE, pesen el {100 * (pitjors ** 2).sum() / (errors_test ** 2).sum():.0f}% de la suma d'errors al quadrat.")

Quatre pingüins, un 6% del total, són responsables d'un 16% del MAE però d'un
**31% del RMSE**: gairebé el doble de pes. Això és el que vol dir que el RMSE
*castiga els errors grans*, vist amb dades reals i no amb un exemple inventat.

Una regla pràctica per llegir-los junts: **divideix el RMSE entre el MAE**.

- Si surt **al voltant d'1,2 o 1,3**, els errors estan repartits de manera
  natural: molts de petits i pocs de grans, com és d'esperar. És el nostre cas:
  356 / 288 = 1,24.
- Si surt **1,5 o més**, hi ha uns quants errors enormes que dominen el RMSE. Val
  la pena anar-los a buscar abans de donar el model per bo.

Compte: **aquests quatre pingüins no s'han de treure** de les dades. Són animals
reals que el model no sap predir bé tenint només l'aleta. Mirar-los serveix per
entendre el model, no per maquillar-ne les mètriques. A l'exercici 4 esbrinaràs
què tenen d'especial.

#### Quan l'error gros és a les dades

Hi ha un altre motiu, molt més comú del que sembla, pel qual el RMSE es dispara:
**una dada mal apuntada**. Simulem-ho. Agafem el primer pingüí de prova i fem com
si algú hagués escrit la seva massa amb un zero de més (46.000 g en lloc de
4.600 g).

In [ ]:
y_test_amb_errada = y_test.copy()
y_test_amb_errada.iloc[0] = y_test_amb_errada.iloc[0] * 10   # un zero de més

for nom, y_real in [("dades correctes", y_test), ("amb un zero de més", y_test_amb_errada)]:
    mae = mean_absolute_error(y_real, pred_test)
    rmse = np.sqrt(mean_squared_error(y_real, pred_test))
    print(f"{nom:18}  MAE = {mae:5.0f} g   RMSE = {rmse:5.0f} g   RMSE / MAE = {rmse / mae:.1f}")

Un sol pingüí mal apuntat d'entre 69 multiplica el MAE per tres, però el RMSE
**el multiplica per catorze**, i el quocient RMSE / MAE passa d'1,2 a més de 5.
Cap model real no s'equivoca d'aquesta manera: quan el quocient es dispara així,
sospita primer de les dades i després del model.

**En resum, per triar entre les dues:**

- Fes servir el **MAE** quan tots els errors et costin el mateix, gram a gram, o
  quan hi hagi valors estranys que no vulguis que ho dominin tot.
- Fes servir el **RMSE** quan un error gran sigui molt pitjor que molts de
  petits, com en l'exemple del medicament.
- **Mira-les sempre totes dues juntes.** La distància entre l'una i l'altra ja és
  una informació que cap de les dues no et dóna per separat.

### 6.5 El R2: comparar-se amb dir sempre la mitjana

El R2 ja el coneixes de la UT1: és el número que dóna `score()`. Ara podem
entendre d'on surt, i ho farem sense fórmules, amb el model de referència.

La idea és aquesta. Agafem el model més ximple possible, el que diu sempre la
massa mitjana, i mirem quant s'equivoca. Després mirem quant s'equivoca el nostre
model. El R2 diu **quina part d'aquell error ens hem estalviat**.

In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.metrics import r2_score

referencia = DummyRegressor(strategy="mean").fit(X_train, y_train)
pred_referencia = referencia.predict(X_test)

error_referencia = mean_squared_error(y_test, pred_referencia)
error_model = mean_squared_error(y_test, pred_test)

print(f"Error del nostre model, com a part del de la referència: {error_model / error_referencia:.2f}")
print(f"Part de l'error que ens hem estalviat:                   {1 - error_model / error_referencia:.2f}")
print(f"R2 segons scikit-learn:                                  {r2_score(y_test, pred_test):.2f}")

El nostre model té només un 22% de l'error del model que diu sempre la mitjana.
Ens n'hem estalviat el 78%, i això és exactament el **R2 = 0,78**.

Així es llegeixen els valors del R2, que ara ja no et semblaran arbitraris:

- **1** vol dir que ens hem estalviat tot l'error: predicció perfecta.
- **0** vol dir que no ens hem estalviat res: fem el mateix que dir la mitjana.
- **Negatiu** vol dir que ho fem *pitjor* que dir la mitjana.

El R2 és molt útil per comparar models sobre les mateixes dades, però té un
defecte: **no diu res en grams**. Un R2 de 0,78 no et diu si t'equivoques 30 g o
3 kg. Per això gairebé sempre l'acompanyarem d'un MAE o d'un RMSE.

### 6.6 Totes les mètriques juntes

Posem-ho tot en una taula, i afegim-hi el model de referència perquè els números
tinguin amb què comparar-se.

In [ ]:
def mesurar(y_real, y_predit):
    return {
        "MAE (g)": mean_absolute_error(y_real, y_predit),
        "desviació (%)": 100 * mean_absolute_percentage_error(y_real, y_predit),
        "RMSE (g)": np.sqrt(mean_squared_error(y_real, y_predit)),
        "R2": r2_score(y_real, y_predit),
    }

taula = pd.DataFrame({
    "referència (mitjana)": mesurar(y_test, pred_referencia),
    "regressió lineal": mesurar(y_test, pred_test),
}).T

taula.round(2)

Aquesta taula resumeix el notebook. Amb una sola mesura, l'aleta, el model redueix
l'error a menys de la meitat del que tindria dient sempre la mitjana: passa de
s'equivocar uns 620 g a uns 290 g, del 15% al 7%.

| Mètrica | Pregunta que respon | Unitats |
|---|---|---|
| **MAE** | De mitjana, quant m'equivoco? | les de la variable (g) |
| **Desviació %** | De mitjana, en quin percentatge m'equivoco? | % |
| **RMSE** | Quant m'equivoco, castigant més els errors grans? | les de la variable (g) |
| **R2** | Quina part de l'error de dir la mitjana m'he estalviat? | sense unitats |

Recorda també el NB 1.2: amb **tres** mesures el R2 sobre aquest mateix test era
de 0,78, i avui, amb **només l'aleta**, en treiem gairebé el mateix. Les dues
mesures del bec aportaven molt poc. Al NB 2.2 estudiarem amb calma què passa
quan afegim variables d'una en una.

## 7. Exercicis

**1. Guanya la recta B a mà.** Torna a la secció 4 i canvia els valors de
`PENDENT_B` i `ORDENADA_B` fins a aconseguir un error mitjà sobre l'entrenament
més petit que el de la recta A. Com t'hi acostes al de scikit-learn (319 g)?
Pista: fixa't en el truc de la secció 4.2 per triar l'ordenada.

**2. Una altra mesura.** Repeteix les seccions 3, 5 i 6 fent servir el gruix del
bec (`bill_depth_mm`) en lloc de l'aleta. Abans d'entrenar, dibuixa el núvol de
punts. Quin signe té el pendent? T'ho esperaves? Quin MAE i quin R2 surten, i
per què creus que són pitjors?

**3. El model amb calculadora.** Fent servir només `model.coef_` i
`model.intercept_` (sense `predict()`), calcula la massa predita per a un pingüí
amb una aleta de 195 mm. Comprova després que `predict()` et dóna el mateix.

**4. El pitjor cas.** Troba el pingüí del conjunt de prova amb l'error més gran.
Quant pesa, quant diu el model i de quina espècie i sexe és? Et sembla que el
model hi podia fer res, tenint només l'aleta?
Pista: `comparacio` té el mateix ordre que `X_test`, i `df.loc[index]` et dóna la
fila completa del pingüí.

**5. Entrenament contra prova.** Calcula el MAE i el RMSE del model sobre
l'entrenament i sobre la prova. S'assemblen? Quina conclusió en treus? Aquesta
comparació serà la protagonista del NB 2.3.

## 8. Per al debat de classe

Una cadena de supermercats vol predir **quantes unitats vendrà cada setmana** de
cada producte per decidir quant n'ha de demanar al proveïdor. Tens un model i has
d'escollir **una** mètrica per explicar-ne els resultats a la direcció.

- Faries servir el MAE, la desviació percentual, el RMSE o el R2? Per què?
- Canvia la resposta si hi ha productes que venen 3 unitats per setmana i
  d'altres que en venen 3.000?
- Què és pitjor per al supermercat: quedar-se sense estoc d'un producte o que
  en sobrin caixes? Afecta això la mètrica que triaries?
- Si haguessis de donar un sol número a la persona que fa les comandes, quin
  seria i com l'hi diries amb una frase?

No hi ha una única resposta correcta: el que s'avalua és que la tria estigui
**justificada pel problema**, no per la mètrica.